In [1]:
# =========================
# 1. IMPORTS (DEDUPLICATED)
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew
from scipy.special import boxcox1p

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error

from category_encoders import TargetEncoder
from xgboost import XGBRegressor


# =========================
# 2. LOAD DATA
# =========================
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv', index_col=0)
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv', index_col=0)

X = train_df.copy()
y = np.log1p(X['SalePrice'])


# =========================
# 3. BASIC PROCESSING
# =========================
X['MSSubClass'] = X['MSSubClass'].astype(str)
X['MoSold'] = X['MoSold'].astype(str)

numerical_cols = X.select_dtypes(include='number').columns.drop('SalePrice')


# =========================
# 4. FEATURE ENGINEERING
# =========================

# --- Skew handling ---
high_skew = [
    'PoolArea','LotArea','3SsnPorch','LowQualFinSF','MiscVal',
    'KitchenAbvGr','BsmtFinSF2','ScreenPorch','BsmtHalfBath',
    'EnclosedPorch','MasVnrArea','OpenPorchSF','LotFrontage',
    'BsmtFinSF1','WoodDeckSF','TotalBsmtSF','1stFlrSF','GrLivArea'
]

for col in high_skew:
    X[col + '_log'] = np.log1p(X[col])


# --- Indicator features ---
indicator_cols = [
    'PoolArea','3SsnPorch','LowQualFinSF','MiscVal',
    'BsmtHalfBath','KitchenAbvGr','ScreenPorch','BsmtFinSF2',
    'EnclosedPorch','BsmtUnfSF','2ndFlrSF'
]

for col in indicator_cols:
    X[col + '_ind'] = (X[col] > 0).astype(int)


# --- Age features ---
X['HouseAge'] = X['YrSold'] - X['YearBuilt']
X['HouseAge_log'] = np.log1p(X['HouseAge'])

X['GarageAge'] = X['YrSold'] - X['GarageYrBlt']
X['GarageAge_log'] = np.log1p(X['GarageAge'])


# --- Binning ---
X['LotArea_bin'] = pd.qcut(X['LotArea'], q=10, labels=False)


# --- Interaction features ---
X['GrLivArea_OverallQual'] = X['GrLivArea'] * X['OverallQual']
X['LotArea_GrLivArea_log'] = np.log1p(X['LotArea']) * np.log1p(X['GrLivArea'])
X['LotArea_GarageArea_sqrt'] = np.sqrt(X['LotArea']) * np.sqrt(X['GarageArea'])


# =========================
# 5. SPLIT
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# =========================
# 6. COLUMN GROUPS
# =========================
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(include=['int64','float64']).columns


# =========================
# 7. PREPROCESSING
# =========================

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols)
])


# =========================
# 8. MODEL
# =========================
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    ))
])


# =========================
# 9. TRAIN + EVALUATE
# =========================
model.fit(X_train, y_train)

preds = model.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, preds))
print("Validation RMSE:", rmse)


# =========================
# 10. CROSS VALIDATION
# =========================
kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    model, X, y,
    cv=kf,
    scoring='neg_root_mean_squared_error'
)

print("CV RMSE:", -np.mean(scores))

/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


Validation RMSE: 0.027153482911322732
CV RMSE: 0.032520706040318245
